In [1]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = "../the_execution_node/data/backtest_5m_5yr.parquet"
MODEL_PATH = "../the_models/meta_labeler_v3.json"
UNIVERSE_PATH = "../the_models/curated_universe.json"

with open(UNIVERSE_PATH, "r") as f:
    baskets = json.load(f).get("baskets", {})

meta_labeler = xgb.Booster()
meta_labeler.load_model(MODEL_PATH)

df = pd.read_parquet(DATA_PATH)
if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)
df = df.sort_index().ffill().dropna()

print(f"[SUCCESS] Lab Ready: {df.shape[0]} rows loaded.")

[SUCCESS] Lab Ready: 501636 rows loaded.


In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from the_research_node.m1_xgboost_trainer import apply_frac_diff, find_optimal_d
from numba import jit

# 1: Precompute all spread data once
print("Precomputing spread signals and features (one-time cost)...\n")

spread_cache = {}

for spread_name, params in baskets.items():
    print(f"Processing spread: {spread_name} ...")
    weights = params.get('weights', {})
    half_life = params.get('half_life', 1.0)
    allocation = params.get('capital_allocation', 0.0)

    if allocation <= 0: continue

    missing = [t for t in weights if t not in df.columns]
    if missing: continue

    spread_val = pd.Series(0.0, index=df.index)
    for ticker, w in weights.items():
        spread_val += df[ticker] * w

    if spread_val.eq(0).all(): continue

    print(f" Computing Z-Scores...")
    # Z-scores matching the trainer
    window = max(int(half_life * 78), 50)
    rolling_mean = spread_val.rolling(window).mean()
    rolling_std = spread_val.rolling(window).std().replace(0, np.nan)
    z_score = (spread_val - rolling_mean) / rolling_std

    print(f" Finding Optimal d (ADF search)...")
    # Fractional diff (expensive ADF search)
    half = len(spread_val) // 2
    opt_d, _ = find_optimal_d(spread_val.iloc[:half])
    spread_fd = apply_frac_diff(spread_val, opt_d)

    print(f"Running meta-labeler predictions...")
    features = pd.DataFrame({
        'frac_diff': spread_fd,
        'volatility': rolling_std,
        'signal_strength': z_score,
    }).dropna()

    valid_idx = features.index
    dmatrix = xgb.DMatrix(features)
    win_probs_array = meta_labeler.predict(dmatrix)
    win_probs = pd.Series(win_probs_array, index=valid_idx)

    vol = spread_val.pct_change().ewm(span=100).std().fillna(0) * np.sqrt(78)

    spread_cache[spread_name] = {
        'spread_val': spread_val.values,
        'z_score': z_score.values,
        'vol': vol.values,
        'allocation': allocation,
        # Store win_probs aligned to df.index (NaN where no prediction)
        'win_probs': win_probs.reindex(df.index).values,
        # Store z_score Series for signal generation at different thresholds
        'z_series': z_score,
    }
    print(f" [CACHED] {spread_name} (d={opt_d:.2f})\n")

print(f"[DONE] Cached {len(spread_cache)} spreads. Monte Carlo will be fast now.\n")

# Convert index times to minutes-since-midnight for Numba
times_minutes = np.array([t.hour * 60 + t.minute for t in df.index.time])

# 2: Numba-compiled trade loop (runs ~100x faster)
@jit(nopython=True)
def run_trades(spread_vals, z_scores, vol_arr, win_probs_arr, times_min, allocation, z_thresh, ai_thresh,
               pt_skew, sl_skew, leverage, warmup=2340):
    # Pure numeric trade simulation — no Python overhead 
    n = len(spread_vals)
    returns = np.zeros(n)
    trades = 0

    in_position = 0
    entry_price = 0.0
    entry_idx = 0
    current_vol = 0.0
    trade_size = 0.0

    for i in range(1, n):
        price = spread_vals[i]
        prev = spread_vals[i - 1]
        z = z_scores[i]
        t_min = times_min[i]
        prob = win_probs_arr[i]

        # Determine raw signal from z-score
        signal = 0
        if z < -z_thresh:
            signal = 1
        elif z > z_thresh:
            signal = -1

        # Filter by meta-labeler confidence
        if np.isnan(prob) or prob <= ai_thresh:
            signal = 0

        if in_position == 0 and i > warmup:
            # Time filter: 09:45 = 585min, 15:45 = 945min
            safe_time = (t_min >= 585) and (t_min <= 945)

            if signal != 0 and safe_time:
                in_position = signal
                entry_price = price
                entry_idx = i
                current_vol = vol_arr[i] if vol_arr[i] > 0 else 0.005
                trades += 1

                # Half-Kelly
                if not np.isnan(prob):
                    kelly = prob - ((1.0 - prob) / 1.5)
                    trade_size = max(0.0, kelly / 2.0)
                else:
                    trade_size = 0.5

                active_cap = allocation * trade_size * leverage
                returns[i] -= active_cap * 0.0005

        elif in_position != 0:
            bars_held = i - entry_idx
            active_cap = allocation * trade_size * leverage

            if prev != 0:
                returns[i] += (price - prev) * in_position * active_cap

            if entry_price != 0:
                trade_ret = (price / entry_price - 1.0) * in_position
            else:
                trade_ret = 0.0

            hit_pt = trade_ret >= (current_vol * pt_skew)
            hit_sl = trade_ret <= -(current_vol * sl_skew)
            hit_time = bars_held >= 120
            hit_mr = (in_position == 1 and z >= 0) or (in_position == -1 and z <= 0)

            if hit_pt or hit_sl or hit_time or hit_mr:
                returns[i] -= active_cap * 0.0005
                in_position = 0
                entry_price = 0.0
                entry_idx = 0
                trade_size = 0.0

    return returns, trades


def simulate_realistic_strategy(z_thresh=2.0, ai_thresh=0.55, pt_skew=1.5, sl_skew=1.0, leverage=20.0):
    # Thin wrapper that loops cached spreads through the Numba kernel 
    total_returns = np.zeros(len(df))
    total_trades = 0
    WARMUP_BARS = 2340

    for name, cache in spread_cache.items():
        ret, trades = run_trades(
            cache['spread_val'], cache['z_score'], cache['vol'],
            cache['win_probs'], times_minutes, cache['allocation'],
            z_thresh, ai_thresh, pt_skew, sl_skew, leverage, WARMUP_BARS
        )
        total_returns += ret
        total_trades += trades

    active = total_returns[WARMUP_BARS:]
    cum = np.cumsum(active)
    max_dd = np.min(cum - np.maximum.accumulate(cum))
    total_pl = cum[-1]

    return total_pl, max_dd, total_trades

Precomputing spread signals and features (one-time cost)...

Processing spread: AVGO_NVDA_Spread ...
 Computing Z-Scores...
 Finding Optimal d (ADF search)...
Running meta-labeler predictions...
 [CACHED] AVGO_NVDA_Spread (d=0.20)

Processing spread: BAC_JPM_Spread ...
 Computing Z-Scores...
 Finding Optimal d (ADF search)...
Running meta-labeler predictions...
 [CACHED] BAC_JPM_Spread (d=0.60)

Processing spread: C_GS_MS_Spread ...
Processing spread: COST_HYG_TLT_WMT_XLU_Spread ...
 Computing Z-Scores...
 Finding Optimal d (ADF search)...
Running meta-labeler predictions...
 [CACHED] COST_HYG_TLT_WMT_XLU_Spread (d=0.20)

Processing spread: CVX_XLE_Spread ...
 Computing Z-Scores...
 Finding Optimal d (ADF search)...
Running meta-labeler predictions...
 [CACHED] CVX_XLE_Spread (d=0.30)

Processing spread: DIS_XLF_Spread ...
 Computing Z-Scores...
 Finding Optimal d (ADF search)...
Running meta-labeler predictions...
 [CACHED] DIS_XLF_Spread (d=0.50)

Processing spread: DUK_SO_Spread ...

In [6]:
import random

num_tests = 50
realistic_results = []

print(f"Running REALISTIC Monte Carlo Search ({num_tests} iterations)...\n")

for i in range(num_tests):
    test_z = round(random.uniform(1.8, 2.5), 2)
    test_ai = round(random.uniform(0.55, 0.68), 2)
    test_pt = round(random.uniform(1.0, 2.0), 2)
    test_sl = round(random.uniform(1.0, 2.0), 2)
    test_lev = round(random.uniform(10.0, 40.0), 1)

    pl, dd, trades = simulate_realistic_strategy(
        z_thresh=test_z, ai_thresh=test_ai,
        pt_skew=test_pt, sl_skew=test_sl, leverage=test_lev
    )

    romd = round(abs(pl / dd), 2) if dd < 0 else (999.99 if pl > 0 else 0)

    realistic_results.append({
        "AI_Conf": test_ai, "Z_Score": test_z,
        "PT_Skew": test_pt, "SL_Skew": test_sl, "Leverage": test_lev,
        "Total P&L": round(pl, 2), "Max Drawdown": round(dd, 2),
        "Trades": trades, "RoMD": romd
    })

realistic_df = pd.DataFrame(realistic_results).sort_values(by="RoMD", ascending=False)
print("=== TOP 10 REALISTIC CONFIGURATIONS (RANKED BY RoMD) ===")
realistic_df.head(10)

Running REALISTIC Monte Carlo Search (50 iterations)...

=== TOP 10 REALISTIC CONFIGURATIONS (RANKED BY RoMD) ===


,AI_Conf,Z_Score,PT_Skew,SL_Skew,Leverage,Total P&L,Max Drawdown,Trades,RoMD
21,0.55,1.96,1.70,1.83,28.5,1317.63,-128.38,12819,10.26
48,0.55,2.04,1.75,1.73,31.2,1410.45,-140.56,12373,10.03
33,0.58,2.18,1.86,1.21,38.5,1593.84,-166.87,11236,9.55
34,0.58,1.98,1.73,1.40,13.2,566.98,-59.58,12277,9.52
18,0.58,1.93,1.32,1.32,20.4,827.17,-88.42,12603,9.36
45,0.56,2.14,1.26,1.90,28.2,1185.32,-127.31,11683,9.31
23,0.59,2.26,1.88,1.03,19.9,802.63,-86.26,10759,9.30
4,0.60,2.04,1.77,1.03,36.5,1470.31,-158.20,11809,9.29
8,0.59,2.21,1.69,1.13,12.7,511.13,-55.05,10960,9.29
3,0.56,2.23,1.24,1.80,13.7,563.20,-61.85,11191,9.11
